# 🎙️ Beatrice V2 Trainer — Custom Dataset

Train your own voice conversion model using Beatrice V2.

**Instructions:**
1. Run **Step 1** to install all modules and mount Google Drive.
2. Run **Option 1** or **Option 2** under **Step 2** to set up your dataset.
3. Run **Step 3** to launch the Settings UI.
4. Run **Step 4** to launch TensorBoard to monitor training metrics.
5. Run **Step 5** to customize your step count and start the training loop.
6. Run **Step 6** to download your VST3 paraphernalia or checkpoint model.
7. Run **Step 7** to optionally clean up all files and free space on Google Drive.

> ⚠️ Make sure you have **GPU runtime** enabled: Runtime → Change runtime type → T4 GPU

In [ ]:
# @title 1. Install Modules & Mount Google Drive
import os
import shutil
from google.colab import drive

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Safely reset directory
%cd /content
if os.path.exists("/content/beatrice-trainer"):
    shutil.rmtree("/content/beatrice-trainer")

!pip install easy-colab-ui-extension > /dev/null 2>&1
from easy_colab_ui_extension.run_and_log import run_and_log
run_and_log("pip install colab-easy-ui2 pyworld==0.3.4", log_file="pip_log.txt", tail_lines=3, expected_lines=500)

# Clone from Hugging Face public repo which contains the full environment
!git clone https://huggingface.co/fierce-cats/beatrice-trainer /content/beatrice-trainer

# Set up paths in Google Drive for persistence
drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
data_dir = os.path.join(drive_base_dir, "data")
output_dir = os.path.join(drive_base_dir, "output")
os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print("Google Drive base folder:", drive_base_dir)
print("✅ Installation complete. Now choose Option 1 or Option 2 below to set up your dataset.")

## 📁 Step 2: Set Up Dataset
Choose either **Option 1** to pull a dataset from Hugging Face, or **Option 2** to upload your own custom files.

In [ ]:
# @title 2a. Option 1: Download Dataset from Hugging Face
huggingface_repo_id = "meoconxinhxan/trump-speech-dataset-tts" # @param {type:"string"}
speaker_name = "trump" # @param {type:"string"}

import os
import glob
import shutil
import soundfile as sf
from datasets import load_dataset, Audio
from huggingface_hub import snapshot_download

# Define paths
drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
data_dir = os.path.join(drive_base_dir, "data", speaker_name)

# Ensure clean data directory
if os.path.exists(data_dir):
    shutil.rmtree(data_dir)
os.makedirs(data_dir, exist_ok=True)

print(f"📥 Downloading dataset from {huggingface_repo_id}...")

# Try loading via load_dataset first (good for parquet/hub audio datasets)
try:
    print("Attempting to load via Hugging Face datasets...")
    ds = load_dataset(huggingface_repo_id, split="train")
    
    # Identify audio column
    audio_col = None
    for col in ds.column_names:
        if isinstance(ds.features[col], Audio) or col in ['audio', 'path']:
            audio_col = col
            break
    
    if audio_col:
        ds = ds.cast_column(audio_col, Audio(sampling_rate=16000))
        print(f"Found audio column: {audio_col}. Extracting WAV files...")
        for i, item in enumerate(ds):
            audio_array = item[audio_col]['array']
            sr = item[audio_col]['sampling_rate']
            target_path = os.path.join(data_dir, f"{speaker_name}_{i}.wav")
            sf.write(target_path, audio_array, sr)
        print(f"✅ Extracted audio files to {data_dir}")
    else:
        raise ValueError("No audio column found in dataset features.")
        
except Exception as e_ds:
    print(f"Could not load as standard HF dataset: {e_ds}")
    print("Attempting direct repository download...")
    try:
        # Try snapshot_download to download wav files directly
        temp_download_dir = "/content/temp_hf_download"
        if os.path.exists(temp_download_dir):
            shutil.rmtree(temp_download_dir)
        
        snapshot_download(
            repo_id=huggingface_repo_id,
            repo_type="dataset",
            local_dir=temp_download_dir,
            local_dir_use_symlinks=False
        )
        
        # Find and copy all WAV files
        wav_files = glob.glob(os.path.join(temp_download_dir, "**/*.wav"), recursive=True)
        if wav_files:
            for i, wf in enumerate(wav_files):
                shutil.copy2(wf, os.path.join(data_dir, f"{speaker_name}_{i}.wav"))
            print(f"✅ Downloaded and copied {len(wav_files)} WAV files to {data_dir}")
        else: 
            # Check if there are ZIP files
            zip_files = glob.glob(os.path.join(temp_download_dir, "**/*.zip"), recursive=True)
            if zip_files:
                import zipfile
                for zf_path in zip_files:
                    print(f"Extracting {os.path.basename(zf_path)}...")
                    with zipfile.ZipFile(zf_path, 'r') as zip_ref:
                        for member in zip_ref.namelist():
                            if member.endswith('/') or member.startswith('__MACOSX') or member.startswith('.'):
                                continue
                            if member.lower().endswith('.wav'):
                                basename = os.path.basename(member)
                                if basename:
                                    shutil.copyfileobj(zip_ref.open(member), open(os.path.join(data_dir, basename), 'wb'))
                print(f"✅ Extracted WAV files from ZIP to {data_dir}")
            else:
                print("❌ No WAV or ZIP files found in the repository.")
    except Exception as e_snap:
        print(f"❌ Failed to download from Hugging Face: {e_snap}")

total_files = len(glob.glob(os.path.join(data_dir, '*.wav')))
print(f"\nSetup complete. Total wav files in {speaker_name} folder: {total_files}")
if total_files > 0:
    print("🎉 Ready to train! Now run the Settings UI or training cell.")

In [ ]:
# @title 2b. Option 2: Upload Your Own Dataset (ZIP of WAV files)
speaker_name = "my_voice" # @param {type:"string"}

import os
import glob
import shutil
import zipfile
from google.colab import files

# Define paths
drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
data_dir = os.path.join(drive_base_dir, "data", speaker_name)

# Ensure clean data directory
if os.path.exists(data_dir):
    shutil.rmtree(data_dir)
os.makedirs(data_dir, exist_ok=True)

print(f"📁 Speaker name: {speaker_name}")
print(f"📂 Data directory: {data_dir}")
print()
print("📤 Please upload a ZIP file containing your WAV audio files or individual WAV files...")
print()

# Upload file
uploaded = files.upload()

if not uploaded:
    print("❌ No file uploaded. Please try again.")
else:
    for filename, content in uploaded.items():
        print(f"📦 Received: {filename} ({len(content) / 1024 / 1024:.1f} MB)")

        if filename.lower().endswith('.zip'):
            # Save and extract ZIP
            zip_path = os.path.join('/content', filename)
            with open(zip_path, 'wb') as f:
                f.write(content)

            print(f"📂 Extracting ZIP to {data_dir}...")
            with zipfile.ZipFile(zip_path, 'r') as zf:
                # Extract all files
                for member in zf.namelist():
                    # Skip directories and hidden files
                    if member.endswith('/') or member.startswith('__MACOSX') or member.startswith('.'):
                        continue
                    # Only extract WAV files
                    if member.lower().endswith('.wav'):
                        basename = os.path.basename(member)
                        if basename:
                            target_path = os.path.join(data_dir, basename)
                            with zf.open(member) as src, open(target_path, 'wb') as dst:
                                dst.write(src.read())
            # Clean up ZIP
            os.remove(zip_path)

        elif filename.lower().endswith('.wav'):
            # Direct WAV upload
            target_path = os.path.join(data_dir, filename)
            with open(target_path, 'wb') as f:
                f.write(content)
        else:
            print(f"⚠️ Skipping {filename} — only ZIP and WAV files accepted.")
            continue

    # Count results
    wav_files = glob.glob(os.path.join(data_dir, '*.wav'))
    total_files = len(wav_files)

    # Calculate total duration if soundfile is available
    try:
        import soundfile as sf
        total_duration = 0
        for wf in wav_files:
            info = sf.info(wf)
            total_duration += info.duration
        duration_str = f"{total_duration / 60:.1f} minutes"
    except:
        duration_str = "unknown"

    print()
    print("=" * 50)
    print(f"✅ Upload complete!")
    print(f"   Speaker:    {speaker_name}")
    print(f"   WAV files:  {total_files}")
    print(f"   Duration:   {duration_str}")
    print("=" * 50)

    if total_files > 0:
        print()
        print("🚀 Ready to train! Now run Step 3 (Start Settings UI).")
    else:
        print()
        print("⚠️ No WAV files found. Make sure your ZIP contains .wav files.")

In [ ]:
# @title 3. Start Settings UI
import os
import torch
from colab_easy_ui.beatrice_trainer_main import wrapped_run_server
from IPython.display import HTML
from IPython.display import display

import portpicker
import time

drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
data_dir = os.path.join(drive_base_dir, "data")
output_dir = os.path.join(drive_base_dir, "output")

def get_device():
  if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
  else:
    device = "cpu"
  return device
device=get_device()

port = portpicker.pick_unused_port()

wrapped_run_server(port=port, colab=True, extract_to=data_dir, extract_to2=output_dir)
time.sleep(2)
from colab_easy_ui.beatrice_trainer_main import get_html_setup
html = get_html_setup(port, device)
display(HTML(html))

In [ ]:
# @title 4. Tensorboard
from google.colab.output import eval_js
import os

drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
output_dir = os.path.join(drive_base_dir, "output")

PORT=18002
os.system(f'tensorboard --logdir={output_dir} --host 0.0.0.0 --port {PORT} &')
proxy = eval_js( "google.colab.kernel.proxyPort(" + str(PORT) + ")" )
print(f"tensorboard launched at {proxy}")

In [ ]:
# @title 5. Start Training
training_steps = 5000 # @param {type:"integer"}
batch_size = 8 # @param {type:"integer"}
learning_rate = 0.00005 # @param {type:"number"}
save_every_n_steps = 1000 # @param {type:"integer"}
warmup_steps = 1000 # @param {type:"integer"}
num_workers = 8 # @param {type:"integer"}
resume_from_checkpoint = True # @param {type:"boolean"}

import json
import os
import shutil
import re
import torchaudio
import sys
import glob

# Force patch the actual torchaudio module file
torchaudio_path = torchaudio.__file__
if "__init__.py" in torchaudio_path:
    with open(torchaudio_path, "r") as f:
        content = f.read()
    if "list_audio_backends" not in content:
        with open(torchaudio_path, "a") as f:
            f.write("\ndef list_audio_backends(): return ['soundfile']\n")

# Ensure paths
drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
working_dir = "/content/beatrice-trainer"
data_dir = os.path.join(drive_base_dir, "data")
output_dir = os.path.join(drive_base_dir, "output")
%cd {working_dir}

# Setup config file
config_path = os.path.join(working_dir, "training_config.json")
config_search = glob.glob("**/default_config.json", recursive=True)
if config_search:
    shutil.copy2(config_search[0], config_path)
    with open(config_path, "r") as f:
        data = json.load(f)
    data["n_steps"] = training_steps
    data["batch_size"] = batch_size
    data["learning_rate_g"] = learning_rate
    data["learning_rate_d"] = learning_rate
    data["save_interval"] = save_every_n_steps
    data["evaluation_interval"] = save_every_n_steps
    data["warmup_steps"] = warmup_steps
    data["num_workers"] = num_workers
    with open(config_path, "w") as f:
        json.dump(data, f, indent=4)
else:
    with open(config_path, 'w') as f:
        json.dump({
            "n_steps": training_steps, 
            "batch_size": batch_size, 
            "learning_rate_g": learning_rate,
            "learning_rate_d": learning_rate,
            "save_interval": save_every_n_steps,
            "evaluation_interval": save_every_n_steps,
            "warmup_steps": warmup_steps,
            "num_workers": num_workers
        }, f)

# Check for latest checkpoint on Google Drive to resume
resume_flag = ""
if resume_from_checkpoint:
    # Find any speaker subdirectories in the output directory
    speaker_dirs = [d for d in glob.glob(os.path.join(output_dir, "*")) if os.path.isdir(d)]
    checkpoint_found = False
    for s_dir in speaker_dirs:
        latest_cp = os.path.join(s_dir, "checkpoint_latest.pt.gz")
        if os.path.exists(latest_cp):
            print(f"🔄 Latest checkpoint found at: {latest_cp}. Resuming training...")
            resume_flag = "-r"
            checkpoint_found = True
            break
    if not checkpoint_found:
        print("⚠️ No checkpoint found to resume from. Starting fresh training...")

print(f"Starting training for {training_steps} steps...")
!PYTHONPATH={working_dir} {sys.executable} -m beatrice_trainer -d {data_dir} -o {output_dir} -c {config_path} {resume_flag}

In [ ]:
# @title 6. Download Output
download_type = "VST3 Paraphernalia (Recommended - Fast)" # @param ["VST3 Paraphernalia (Recommended - Fast)", "Latest Checkpoint (.pt.gz)", "Entire Output Folder (Slow)"]

import os
import glob
import shutil
from google.colab import files

drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
output_path = os.path.join(drive_base_dir, "output")

# Helper function to zip and download
def colab_zip_and_download(folder_to_zip, zip_name):
    zip_file = f"/content/{zip_name}.zip"
    if os.path.exists(zip_file):
        os.remove(zip_file)
    print(f"📦 Zipping {folder_to_zip}...")
    shutil.make_archive(f"/content/{zip_name}", 'zip', folder_to_zip)
    print(f"📥 Downloading {zip_name}.zip...")
    files.download(zip_file)
    print("✅ Download started!")

if not os.path.exists(output_path) or not os.listdir(output_path):
    print("❌ No outputs found in Google Drive! Please run training first.")
else:
    if "VST3 Paraphernalia" in download_type:
        # Find any paraphernalia folders (either directly in output or inside a subfolder)
        paraphernalia_dirs = sorted(glob.glob(os.path.join(output_path, "**/paraphernalia_*"), recursive=True))
        if paraphernalia_dirs:
            latest_p = paraphernalia_dirs[-1]
            print(f"📂 Found latest paraphernalia folder: {latest_p}")
            colab_zip_and_download(latest_p, f"beatrice_vst3_files")
        else:
            print("❌ No paraphernalia folders found. Did training save checkpoints?")
            
    elif "Latest Checkpoint" in download_type:
        # Find any checkpoint_latest.pt.gz (directly or inside a subfolder)
        latest_cp_files = sorted(glob.glob(os.path.join(output_path, "**/checkpoint_latest.pt.gz"), recursive=True))
        if not latest_cp_files:
            # Fallback to any checkpoint_data_*.pt.gz files
            latest_cp_files = sorted(glob.glob(os.path.join(output_path, "**/checkpoint_data_*.pt.gz"), recursive=True))
            
        if latest_cp_files:
            latest_cp = latest_cp_files[-1]
            print(f"📥 Downloading latest checkpoint: {latest_cp}...")
            files.download(latest_cp)
            print("✅ Download started!")
        else:
            print("❌ No checkpoint files (.pt.gz) found.")
            
    else:
        # Zip and download entire output folder
        colab_zip_and_download(output_path, f"beatrice_all_outputs")

In [ ]:
# @title 7. Delete Dataset and Output Files (Clean up)
# @markdown **WARNING:** This will permanently delete your dataset and training outputs from Google Drive!
confirm_deletion = False # @param {type:"boolean"}

import os
import shutil

if not confirm_deletion:
    print("⚠️ Please check 'confirm_deletion' checkbox to delete the files.")
else:
    drive_base_dir = "/content/drive/MyDrive/Beatrice_Trainer"
    
    # Delete local beatrice-trainer
    if os.path.exists("/content/beatrice-trainer"):
         print("🗑️ Deleting local repo...")
         shutil.rmtree("/content/beatrice-trainer")
         
    # Delete dataset folder
    data_dir = os.path.join(drive_base_dir, "data")
    if os.path.exists(data_dir):
        print(f"🗑️ Deleting dataset from Google Drive: {data_dir}")
        shutil.rmtree(data_dir)
        
    # Delete output folder
    output_dir = os.path.join(drive_base_dir, "output")
    if os.path.exists(output_dir):
        print(f"🗑️ Deleting outputs from Google Drive: {output_dir}")
        shutil.rmtree(output_dir)
        
    print("✨ Cleanup complete! All files deleted.")